# Bronze Layer - Raw Data Ingestion

The Bronze layer is the first layer of the Medallion Architecture.

Its main responsibility is to ingest raw source data and store it as Delta tables with minimal processing.

In this implementation:
- Data is loaded directly from raw source files.
- Delta Live Tables (DLT) is not used.
- Fivetran is not used.
- PySpark DataFrame API is used for all ingestion processes.

Bronze layer responsibilities:
- Read raw files
- Preserve source data structure
- Apply minimal ingestion-level processing
- Add ingestion metadata
- Store raw data as Delta tables

## Environment Setup

Import required PySpark libraries and configure the Bronze layer.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp


LAYER = "bronze"

## Utility Methods

A reusable ingestion function is created to standardize loading raw datasets.

This function:
- Reads CSV and JSON files
- Removes unnecessary technical columns
- Adds ingestion timestamp
- Returns a DataFrame ready for Bronze storage

In [0]:
def ingest_raw_data(path: str, file_format: str = "csv") -> DataFrame:
    """
    Load raw source data into Bronze layer.

    Parameters
    ----------
    path : str
        Raw dataset path.

    file_format : str
        Source file format.

    Returns
    -------
    DataFrame
        Ingested raw dataframe.
    """

    if file_format == "csv":

        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(path)
        )

    elif file_format == "json":

        df = spark.read.json(path)

    else:
        raise ValueError(
            "Unsupported file format"
        )


    # Remove technical columns if present
    cols = [
        c for c in df.columns
        if not c.startswith("_")
    ]

    df = df.select(*cols)


    # Add ingestion metadata
    df = df.withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )


    return df

## Policies - MySQL Source

Policy records are loaded from the raw MySQL sample dataset.

Source:
data/samples/mysql/policies.csv

Bronze processing:
- Read CSV file
- Preserve original attributes
- Add ingestion timestamp
- Store as Delta table

## Policies - MySQL Source

Policy records are loaded from the raw MySQL sample dataset.

Source:
data/samples/mysql/policies.csv

Bronze processing:
- Read CSV file
- Preserve original attributes
- Add ingestion timestamp
- Store as Delta table

In [0]:
policies_df = ingest_raw_data(
    path="/Workspace/Users/wangwangming1975@gmail.com/insurance-claims/data/samples/mysql/policies.csv",
    file_format="csv"
)


(
    policies_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("bronze_policies")
)

## Claims - MongoDB Source

Claim records are loaded from the raw MongoDB sample dataset.

Source:
data/samples/mongodb/claims.json

Bronze processing:
- Read JSON file
- Preserve source structure
- Add ingestion timestamp
- Store as Delta table

In [0]:
claims_df = (
    spark.read
    .option("multiline", True)
    .json(
        "file:/Workspace/Users/wangwangming1975@gmail.com/insurance-claims/data/samples/s3/tmp/claims.json"
    )
)


# Remove technical columns if present
cols = [
    c for c in claims_df.columns
    if not c.startswith("_")
]

claims_df = claims_df.select(*cols)


# Add ingestion metadata
from pyspark.sql.functions import current_timestamp

claims_df = claims_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)


(
    claims_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("bronze_claims")
)

## Traffic Accidents - S3 Source

Accident records are loaded from the raw S3 sample dataset.

Source:
data/samples/s3/accidents.csv

Bronze processing:
- Read CSV file
- Preserve raw attributes
- Add ingestion timestamp
- Store as Delta table

In [0]:
accidents_df = ingest_raw_data(
    path="file:/Workspace/Users/wangwangming1975@gmail.com/insurance-claims/data/samples/s3/external/accidents.csv.gz",
    file_format="csv"
)


(
    accidents_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("bronze_accidents")
)

# Bronze Layer Validation

Validate that all Bronze Delta tables have been successfully created.

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
spark.table("bronze_policies").printSchema()

In [0]:
spark.table("bronze_claims").printSchema()

In [0]:
spark.table("bronze_accidents").printSchema()

In [0]:
%sh
head -5 /Workspace/Users/wangwangming1975@gmail.com/insurance-claims/data/samples/s3/tmp/claims.json

In [0]:
display(
    spark.table("bronze_policies").limit(5)
)

In [0]:
display(
    spark.table("bronze_claims").limit(5)
)

In [0]:
display(
    spark.table("bronze_accidents").limit(5)
)